In [1]:
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, move="", g=0, h=0):
        self.board = board  # 1D tuple representing the 3x3 board (0 for empty tile)
        self.parent = parent
        self.move = move
        self.g = g  # Cost from start to current node
        self.h = h  # Heuristic cost to goal
        self.f = g + h

    def __lt__(self, other):
        return self.f < other.f

    def get_neighbors(self):
        neighbors = []
        zero_idx = self.board.index(0)
        row, col = zero_idx // 3, zero_idx % 3

        # Define possible movements of the empty tile (0)
        moves = {
            "UP": (-1, 0),
            "DOWN": (1, 0),
            "LEFT": (0, -1),
            "RIGHT": (0, 1)
        }

        for direction, (dr, dc) in moves.items():
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_zero_idx = new_row * 3 + new_col
                new_board = list(self.board)
                # Swap 0 with the target tile
                new_board[zero_idx], new_board[new_zero_idx] = new_board[new_zero_idx], new_board[zero_idx]
                neighbors.append((tuple(new_board), direction))
        return neighbors

# Heuristic 1: Manhattan Distance
def manhattan_distance(board, goal):
    distance = 0
    for i, tile in enumerate(board):
        if tile != 0:
            goal_idx = goal.index(tile)
            curr_row, curr_col = i // 3, i % 3
            goal_row, goal_col = goal_idx // 3, goal_idx % 3
            distance += abs(curr_row - goal_row) + abs(curr_col - goal_col)
    return distance

# Heuristic 2: Misplaced Tiles
def misplaced_tiles(board, goal):
    count = 0
    for i, tile in enumerate(board):
        if tile != 0 and tile != goal[i]:
            count += 1
    return count

# A* Search Algorithm
def a_star(start_board, goal_board, heuristic_type="manhattan"):
    heuristic_fn = manhattan_distance if heuristic_type == "manhattan" else misplaced_tiles

    start_state = PuzzleState(
        board=start_board,
        g=0,
        h=heuristic_fn(start_board, goal_board)
    )

    open_set = []
    heapq.heappush(open_set, start_state)
    closed_set = set()

    nodes_expanded = 0

    while open_set:
        current_state = heapq.heappop(open_set)

        if current_state.board == goal_board:
            path = []
            curr = current_state
            while curr.parent:
                path.append((curr.move, curr.board))
                curr = curr.parent
            return path[::-1], nodes_expanded

        closed_set.add(current_state.board)
        nodes_expanded += 1

        for neighbor_board, move in current_state.get_neighbors():
            if neighbor_board in closed_set:
                continue

            g_score = current_state.g + 1
            h_score = heuristic_fn(neighbor_board, goal_board)
            neighbor_state = PuzzleState(
                board=neighbor_board,
                parent=current_state,
                move=move,
                g=g_score,
                h=h_score
            )

            # Simple check to avoid redundant states in open_set
            heapq.heappush(open_set, neighbor_state)

    return None, nodes_expanded

def print_board(board):
    for i in range(0, 9, 3):
        print(list(board[i:i+3]))
    print()

# --- Example Usage ---
if __name__ == "__main__":
    # 0 represents the empty space
    start = (1, 2, 3, 0, 4, 6, 7, 5, 8)
    goal = (1, 2, 3, 4, 5, 6, 7, 8, 0)

    print("Start State:")
    print_board(start)

    # Run with Manhattan Heuristic
    print("--- Solving with Manhattan Heuristic ---")
    path, expanded_mh = a_star(start, goal, heuristic_type="manhattan")
    if path:
        print(f"Solved in {len(path)} steps. Nodes expanded: {expanded_mh}")
        for move, board in path:
            print(f"Move: {move}")
            print_board(board)
    else:
        print("No solution found.")

    # Run with Misplaced Tiles Heuristic
    print("--- Solving with Misplaced Tiles Heuristic ---")
    path, expanded_mt = a_star(start, goal, heuristic_type="misplaced")
    if path:
        print(f"Solved in {len(path)} steps. Nodes expanded: {expanded_mt}")
    else:
        print("No solution found.")

Start State:
[1, 2, 3]
[0, 4, 6]
[7, 5, 8]

--- Solving with Manhattan Heuristic ---
Solved in 3 steps. Nodes expanded: 3
Move: RIGHT
[1, 2, 3]
[4, 0, 6]
[7, 5, 8]

Move: DOWN
[1, 2, 3]
[4, 5, 6]
[7, 0, 8]

Move: RIGHT
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]

--- Solving with Misplaced Tiles Heuristic ---
Solved in 3 steps. Nodes expanded: 3
